# GeoAI – Geolocation Estimation from Street View Images

Dieses Notebook trainiert ein CNN-Modell (Transfer Learning mit EfficientNet),
das anhand eines Fotos vorhersagt, **in welchem Land und an welchem Ort** das Bild aufgenommen wurde.

**Ansatz:**
1. Welt in geografische Gitterzellen einteilen
2. CNN klassifiziert jedes Bild in eine Zelle
3. Vorhersage → Lat/Lon (Zellmittelpunkt) → Reverse Geocoding → Land + Stadt
4. Ergebnis vs. Ground Truth vergleichen

**Dataset:** paulchambaz/google-street-view – 10.000 Street-View-Bilder weltweit

## 1. Imports & Konfiguration

In [ ]:
import os, math, json, time, random, warnings
from pathlib import Path

# Warnungen unterdrücken (HuggingFace Symlink-Warnung auf Windows)
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings('ignore', category=UserWarning, module='huggingface_hub')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm   # auto: funktioniert in Notebook + Terminal, mit oder ohne ipywidgets

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
from sklearn.model_selection import train_test_split

# ── Reproduzierbarkeit ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Gerät : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
else:
    print('Hinweis: Training auf CPU – dauert länger. GPU empfohlen.')

In [ ]:
# ── Konfiguration (hier anpassen) ──────────────────────────────────────────
CFG = {
    'grid_deg'          : 10.0,
    'min_imgs_per_cell' : 3,

    # Modell
    'backbone'          : 'efficientnet_b0',
    'img_size'          : 224,
    'dropout'           : 0.5,       # erhöht von 0.3 → stärker regularisiert

    # Training
    'batch_size'        : 32,
    'epochs'            : 30,        # mehr Epochen, da Phase 1 langsamer lernt
    'lr'                : 3e-4,
    'lr_finetune'       : 5e-5,      # niedrige LR für Backbone beim Finetuning
    'weight_decay'      : 1e-3,      # erhöht von 1e-4
    'val_split'         : 0.15,
    'test_split'        : 0.10,

    # Zweiphasen-Training:
    # Phase 1 (Epochen 1..freeze_epochs): nur Kopf trainieren, Backbone eingefroren
    # Phase 2 (danach): ganzes Netz finetunen mit niedrigerem LR
    'freeze_epochs'     : 8,

    # Checkpoint-Verhalten
    # False → Training fortsetzen/überspringen wenn best_model.pth existiert
    # True  → immer neu trainieren
    'force_retrain'     : True,      # auf False setzen nach fertigem Training

    # Pfade
    'checkpoint_dir'    : 'checkpoints',
    'best_model'        : 'checkpoints/best_model.pth',
    'cell_mapping'      : 'checkpoints/cell_mapping.json',
    'history_file'      : 'checkpoints/training_history.json',
}

Path(CFG['checkpoint_dir']).mkdir(exist_ok=True)
print('Konfiguration geladen.')
if Path(CFG['best_model']).exists():
    print('→ Checkpoint vorhanden. force_retrain =', CFG['force_retrain'])
else:
    print('→ Kein Checkpoint – Training startet neu.')

## 2. Dataset herunterladen

In [ ]:
import kagglehub

DATASET_PATH = Path(kagglehub.dataset_download('paulchambaz/google-street-view'))
IMG_DIR      = DATASET_PATH / 'dataset'
CSV_PATH     = IMG_DIR / 'coords.csv'

print(f'Dataset : {DATASET_PATH}')
print(f'Bilder  : {len(list(IMG_DIR.glob("*.png"))):,}')
print(f'CSV     : {CSV_PATH}')

## 3. Metadaten laden & aufbereiten

Das CSV hat **keinen echten Header** – die erste Zeile enthält bereits Koordinaten für Bild 0.

In [ ]:
# header=None → pandas liest ALLE Zeilen als Daten
df_raw = pd.read_csv(CSV_PATH, header=None, names=['lat', 'lon'])

print(f'Einträge im CSV : {len(df_raw):,}')
print(df_raw.head(3))

# Index entspricht dem Dateinamen (Bild 0 → 0.png, Bild 1 → 1.png, ...)
df_raw['img_path'] = df_raw.index.map(lambda i: str(IMG_DIR / f'{i}.png'))

# Nur Bilder die tatsächlich existieren
df_raw['exists'] = df_raw['img_path'].apply(lambda p: Path(p).exists())
df_meta = df_raw[df_raw['exists']].drop(columns='exists').reset_index(drop=True)

print(f'Bilder mit Koordinaten: {len(df_meta):,}')
print(f'Lat-Bereich: {df_meta["lat"].min():.2f} bis {df_meta["lat"].max():.2f}')
print(f'Lon-Bereich: {df_meta["lon"].min():.2f} bis {df_meta["lon"].max():.2f}')

In [ ]:
# Geografische Verteilung visualisieren
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_facecolor('#cce5ff')
ax.scatter(df_meta['lon'], df_meta['lat'], s=3, alpha=0.5, c='darkred')
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_title(f'Geografische Verteilung der {len(df_meta):,} Bilder', fontsize=13)
ax.set_xlabel('Längengrad'); ax.set_ylabel('Breitengrad')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Geografische Gitterzellen erstellen

In [ ]:
STEP = CFG['grid_deg']

def coords_to_cell(lat: float, lon: float) -> str:
    lat_bin = math.floor(lat / STEP) * STEP
    lon_bin = math.floor(lon / STEP) * STEP
    return f'{lat_bin:.1f}_{lon_bin:.1f}'

def cell_center(cell_id: str):
    lat_bin, lon_bin = map(float, cell_id.split('_'))
    return lat_bin + STEP / 2, lon_bin + STEP / 2

df_meta['cell_id'] = df_meta.apply(lambda r: coords_to_cell(r['lat'], r['lon']), axis=1)

# Seltene Zellen entfernen
cell_counts = df_meta['cell_id'].value_counts()
valid_cells  = cell_counts[cell_counts >= CFG['min_imgs_per_cell']].index
df_meta      = df_meta[df_meta['cell_id'].isin(valid_cells)].reset_index(drop=True)

# Klassen-Index
unique_cells = sorted(df_meta['cell_id'].unique())
cell_to_idx  = {c: i for i, c in enumerate(unique_cells)}
idx_to_cell  = {i: c for c, i in cell_to_idx.items()}
NUM_CLASSES  = len(unique_cells)

df_meta['label'] = df_meta['cell_id'].map(cell_to_idx)

print(f'Klassen (Zellen) : {NUM_CLASSES}')
print(f'Bilder (gefiltert): {len(df_meta):,}')
print(f'Bilder/Zelle (Ø) : {len(df_meta)/NUM_CLASSES:.1f}')

# Mapping speichern (für spätere Inference ohne Retraining)
with open(CFG['cell_mapping'], 'w') as f:
    json.dump({'cell_to_idx': cell_to_idx,
               'idx_to_cell': {str(k): v for k, v in idx_to_cell.items()},
               'step': STEP}, f)
print('Zellen-Mapping gespeichert.')

## 5. Train / Val / Test Split

In [ ]:
# Klassen mit zu wenigen Bilder für Stratify rausfiltern
label_counts = df_meta['label'].value_counts()
ok_labels    = label_counts[label_counts >= 3].index
df_strat     = df_meta[df_meta['label'].isin(ok_labels)]

df_trainval, df_test = train_test_split(
    df_strat, test_size=CFG['test_split'],
    stratify=df_strat['label'], random_state=SEED
)
val_frac = CFG['val_split'] / (1 - CFG['test_split'])

# Für val-split: Klassen mit < 2 Bildern ebenfalls entfernen
label_counts2 = df_trainval['label'].value_counts()
ok_labels2    = label_counts2[label_counts2 >= 2].index
df_trainval2  = df_trainval[df_trainval['label'].isin(ok_labels2)]

df_train, df_val = train_test_split(
    df_trainval2, test_size=val_frac,
    stratify=df_trainval2['label'], random_state=SEED
)

print(f'Train : {len(df_train):,}')
print(f'Val   : {len(df_val):,}')
print(f'Test  : {len(df_test):,}')

## 6. PyTorch Dataset & DataLoader

In [ ]:
SZ       = CFG['img_size']
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((SZ + 48, SZ + 48)),
    transforms.RandomCrop(SZ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),                                    # neu: Rotation
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.3, hue=0.08),                # stärker
    transforms.RandomGrayscale(p=0.1),                               # neu: manchmal SW
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),            # neu: Teile löschen
])

val_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(IMG_MEAN, IMG_STD),
])

class GeoDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row['img_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['label'])
        lat, lon = float(row['lat']), float(row['lon'])
        return img, label, lat, lon

train_ds = GeoDataset(df_train, train_tf)
val_ds   = GeoDataset(df_val,   val_tf)
test_ds  = GeoDataset(df_test,  val_tf)

NW = 0
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=NW)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=NW)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=NW)

print(f'Train-Batches : {len(train_loader)}')
print(f'Val-Batches   : {len(val_loader)}')
print('Augmentierung: Rotation, ColorJitter, Grayscale, RandomErasing aktiv.')

# Beispiel-Bilder
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, (_, row) in zip(axes.flat, df_train.sample(8, random_state=1).iterrows()):
    try:
        ax.imshow(Image.open(row['img_path']).convert('RGB'))
        ax.set_title(f'{row["lat"]:.1f}°, {row["lon"]:.1f}°', fontsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, str(e), ha='center', fontsize=6)
    ax.axis('off')
plt.suptitle('Beispiel-Trainingsbilder', fontsize=12)
plt.tight_layout(); plt.show()

## 7. Modell-Architektur

In [ ]:
class GeoModel(nn.Module):
    """Pretrained CNN-Backbone + Klassifikationskopf für geografische Zellen."""

    def __init__(self, backbone_name: str, num_classes: int, dropout: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


model = GeoModel(CFG['backbone'], NUM_CLASSES, CFG['dropout']).to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Backbone   : {CFG["backbone"]}')
print(f'Klassen    : {NUM_CLASSES}')
print(f'Parameter  : {total:,} gesamt, {trainable:,} trainierbar')

## 8. Distanz-Metriken (Haversine)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def batch_haversine(pred_idxs, true_lats, true_lons):
    return [
        haversine_km(*cell_center(idx_to_cell[int(p)]), float(tl), float(tn))
        for p, tl, tn in zip(pred_idxs, true_lats, true_lons)
    ]

# Präzisionsthresholds (wie im ECCV-Paper)
THRESHOLDS = {'Straße': 1, 'Stadt': 25, 'Region': 200, 'Land': 750, 'Kontinent': 2500}
print('Haversine-Distanz & Schwellwerte bereit.')

## 9. Training

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.15)

def set_backbone_frozen(frozen: bool):
    """Backbone ein- oder ausfrieren."""
    for param in model.backbone.parameters():
        param.requires_grad = not frozen
    status = 'eingefroren' if frozen else 'aktiv'
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Backbone {status} | Trainierbare Parameter: {trainable:,}')

def make_optimizer(finetune=False):
    lr = CFG['lr_finetune'] if finetune else CFG['lr']
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=CFG['weight_decay']
    )

def run_epoch(loader, optimizer, mode='train'):
    is_train = (mode == 'train')
    model.train(is_train)
    total_loss, total_correct, all_dists = 0.0, 0, []
    with torch.set_grad_enabled(is_train):
        for imgs, labels, t_lats, t_lons in tqdm(loader, desc=mode, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_loss    += loss.item() * len(imgs)
            all_dists.extend(batch_haversine(preds.cpu(), t_lats, t_lons))
    n = len(loader.dataset)
    return {'loss': total_loss/n, 'acc': total_correct/n,
            'median_km': np.median(all_dists), 'mean_km': np.mean(all_dists)}


def do_training(start_epoch, best_val_loss, history):
    print(f'Epochen {start_epoch}–{CFG["epochs"]} | Klassen: {NUM_CLASSES} | Gerät: {DEVICE}')
    print(f'Phase 1 (Backbone eingefroren): Epochen 1–{CFG["freeze_epochs"]}')
    print(f'Phase 2 (Finetuning):           Epochen {CFG["freeze_epochs"]+1}–{CFG["epochs"]}\n')

    optimizer = None
    scheduler = None
    current_phase = None

    for epoch in range(start_epoch, CFG['epochs'] + 1):
        # ── Phasenwechsel ──────────────────────────────────────────────
        if epoch <= CFG['freeze_epochs']:
            if current_phase != 1:
                current_phase = 1
                print(f'\n── Phase 1: Backbone einfrieren ──')
                set_backbone_frozen(True)
                optimizer = make_optimizer(finetune=False)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=CFG['freeze_epochs'], eta_min=1e-5)
        else:
            if current_phase != 2:
                current_phase = 2
                remaining = CFG['epochs'] - CFG['freeze_epochs']
                print(f'\n── Phase 2: Finetuning (LR={CFG["lr_finetune"]}) ──')
                set_backbone_frozen(False)
                optimizer = make_optimizer(finetune=True)
                scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=remaining, eta_min=1e-6)

        t0  = time.time()
        trn = run_epoch(train_loader, optimizer, 'train')
        val = run_epoch(val_loader,   optimizer, 'val')
        scheduler.step()

        saved = ''
        if val['loss'] < best_val_loss:
            best_val_loss = val['loss']
            torch.save({
                'epoch': epoch, 'model_state': model.state_dict(),
                'val_loss': best_val_loss, 'num_classes': NUM_CLASSES, 'cfg': CFG,
            }, CFG['best_model'])
            saved = ' [gespeichert]'

        row = {'epoch': epoch, **{f't_{k}': v for k, v in trn.items()},
                               **{f'v_{k}': v for k, v in val.items()}}
        history.append(row)
        print(f'Ep {epoch:02d}/{CFG["epochs"]} | '
              f'Loss {trn["loss"]:.4f}/{val["loss"]:.4f} | '
              f'Acc {100*trn["acc"]:.1f}/{100*val["acc"]:.1f}% | '
              f'MedianKm {val["median_km"]:,.0f} | '
              f'{time.time()-t0:.0f}s{saved}')

    return history


# ── Checkpoint prüfen ────────────────────────────────────────────────────────
history = []

if Path(CFG['best_model']).exists() and not CFG['force_retrain']:
    ckpt = torch.load(CFG['best_model'], map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    saved_epoch   = ckpt['epoch']
    best_val_loss = ckpt['val_loss']

    if Path(CFG['history_file']).exists():
        with open(CFG['history_file']) as f:
            history = json.load(f)

    if saved_epoch >= CFG['epochs']:
        print(f'Training bereits abgeschlossen ({saved_epoch}/{CFG["epochs"]} Epochen).')
        print(f'  Val-Loss : {best_val_loss:.4f}  |  Klassen : {ckpt["num_classes"]}')
        print('Setze force_retrain=True um neu zu starten.')
    else:
        print(f'Checkpoint gefunden – setze Training fort ab Epoche {saved_epoch + 1}.')
        history = do_training(saved_epoch + 1, best_val_loss, history)
        with open(CFG['history_file'], 'w') as f:
            json.dump(history, f)
        print('Training abgeschlossen.')
else:
    if CFG['force_retrain']:
        print('force_retrain=True → starte von Epoche 1.')
    else:
        print('Kein Checkpoint – starte Training.')
    history = do_training(1, float('inf'), history)
    with open(CFG['history_file'], 'w') as f:
        json.dump(history, f)
    print('Training abgeschlossen.')

df_history = pd.DataFrame(history)

train:   0%|          | 0/234 [00:00<?, ?it/s]

val:   0%|          | 0/47 [00:00<?, ?it/s]

Ep 16/20 | Loss 0.9773/2.9562 | Acc 99.7/43.9% | MedianKm 686 | 1166s


train:   0%|          | 0/234 [00:00<?, ?it/s]

In [ ]:
# Trainingskurven (nur wenn History vorhanden)
if df_history.empty:
    print('Keine Trainingshistorie vorhanden – Kurven-Plot übersprungen.')
    print('(History wird beim nächsten Training automatisch gespeichert.)')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(df_history['epoch'], df_history['t_loss'], label='Train')
    axes[0].plot(df_history['epoch'], df_history['v_loss'], label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    axes[1].plot(df_history['epoch'], df_history['t_acc']*100, label='Train')
    axes[1].plot(df_history['epoch'], df_history['v_acc']*100, label='Val')
    axes[1].set_title('Genauigkeit (%)'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    axes[2].plot(df_history['epoch'], df_history['v_median_km'], color='darkorange')
    axes[2].set_title('Val Median-Distanz (km)'); axes[2].set_xlabel('Epoch')

    plt.suptitle('Trainingsverlauf', fontsize=13)
    plt.tight_layout(); plt.show()

## 10. Evaluation auf Test-Set

In [ ]:
# Bestes Modell laden
ckpt = torch.load(CFG['best_model'], map_location=DEVICE, weights_only=True)
model.load_state_dict(ckpt['model_state'])
print(f'Bestes Modell geladen (Epoch {ckpt["epoch"]}, Val-Loss {ckpt["val_loss"]:.4f})')

model.eval()
all_preds, all_labels, all_lats, all_lons = [], [], [], []

with torch.no_grad():
    for imgs, labels, t_lats, t_lons in tqdm(test_loader, desc='Test'):
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_lats.extend(t_lats.numpy())
        all_lons.extend(t_lons.numpy())

all_dists = np.array(batch_haversine(all_preds, all_lats, all_lons))
test_acc  = np.mean(np.array(all_preds) == np.array(all_labels))

print(f'\n--- Test-Ergebnisse ---')
print(f'Zell-Genauigkeit : {100*test_acc:.2f}%')
print(f'Median-Distanz   : {np.median(all_dists):,.0f} km')
print(f'Mittlere Distanz : {np.mean(all_dists):,.0f} km')
print()
print('Präzision @ Schwellwert:')
for name, km in THRESHOLDS.items():
    pct = 100 * np.mean(all_dists < km)
    bar = '█' * int(pct / 2)
    print(f'  {name:<12} (<{km:>5} km) : {pct:5.1f}%  {bar}')

In [ ]:
# Distanz-Histogramm + GCD-Kurve
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(all_dists, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(np.median(all_dists), color='red', linestyle='--',
                label=f'Median: {np.median(all_dists):,.0f} km')
axes[0].set_xlabel('Distanz (km)'); axes[0].set_title('Fehlerverteilung (Test-Set)')
axes[0].legend()

sd  = np.sort(all_dists)
cdf = np.arange(1, len(sd)+1) / len(sd)
axes[1].plot(sd, cdf*100, linewidth=2)
for name, km in THRESHOLDS.items():
    pct = 100 * np.mean(all_dists < km)
    axes[1].axvline(km, linestyle=':', alpha=0.6, color='gray')
    axes[1].text(km*1.05, pct-4, f'{name}\n{pct:.0f}%', fontsize=7, color='darkred')
axes[1].set_xscale('log')
axes[1].set_xlabel('Distanz (km, log)'); axes[1].set_ylabel('% korrekt')
axes[1].set_title('Kumulative Präzisionskurve (GCD)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 11. Reverse Geocoding

Koordinaten → Land, Stadt via **Nominatim** (kostenlos, kein API-Key nötig)

In [ ]:
geolocator = Nominatim(user_agent='geoai_v1', timeout=10)
_geo_cache: dict = {}  # lokaler Cache um Nominatim-Limit zu schonen

def reverse_geocode(lat: float, lon: float) -> dict:
    key = (round(lat, 2), round(lon, 2))
    if key in _geo_cache:
        return _geo_cache[key]

    for _ in range(3):
        try:
            loc  = geolocator.reverse((lat, lon), language='de', zoom=10)
            if loc is None:
                result = {'display': 'Unbekannt', 'country': '?', 'city': '?'}
            else:
                addr = loc.raw.get('address', {})
                city = (addr.get('city') or addr.get('town') or
                        addr.get('village') or addr.get('county') or '?')
                result = {'display': loc.address,
                          'country': addr.get('country', '?'),
                          'city':    city}
            _geo_cache[key] = result
            time.sleep(1.1)  # Nominatim-Limit: 1 Request/s
            return result
        except GeocoderTimedOut:
            time.sleep(2)

    result = {'display': 'Fehler', 'country': '?', 'city': '?'}
    _geo_cache[key] = result
    return result

# Kurzer Test
r = reverse_geocode(48.8566, 2.3522)
print('Test (Paris):', r)

## 12. Einzelbild-Vorhersage: Vorhersage vs. Wahrheit

In [ ]:
def predict_image(img_path: str, true_lat: float = None, true_lon: float = None, top_k: int = 3):
    """
    Zeigt Vorhersage für ein Bild:
    - Vorhergesagtes Land & Stadt (via Reverse Geocoding)
    - Vergleich mit Ground Truth (falls angegeben)
    - Distanz in km
    """
    img    = Image.open(img_path).convert('RGB')
    tensor = val_tf(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor)[0], dim=0)

    top_probs, top_idxs = probs.topk(top_k)
    top_probs = top_probs.cpu().numpy()
    top_idxs  = top_idxs.cpu().numpy()

    # Beste Vorhersage
    pred_lat, pred_lon = cell_center(idx_to_cell[top_idxs[0]])
    pred_geo = reverse_geocode(pred_lat, pred_lon)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Eingabebild', fontsize=11)

    ax = axes[1]
    ax.set_facecolor('#cce5ff')
    ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
    ax.set_xlabel('Längengrad'); ax.set_ylabel('Breitengrad')
    ax.grid(True, alpha=0.3)

    ax.scatter(pred_lon, pred_lat, c='red', s=300, zorder=6, marker='*', label='Vorhersage')

    dist_km = None
    true_geo = None
    if true_lat is not None and true_lon is not None:
        ax.scatter(true_lon, true_lat, c='limegreen', s=200, zorder=6, marker='D', label='Wahrheit')
        ax.plot([pred_lon, true_lon], [pred_lat, true_lat], 'k--', linewidth=1, alpha=0.7)
        dist_km  = haversine_km(pred_lat, pred_lon, true_lat, true_lon)
        true_geo = reverse_geocode(true_lat, true_lon)

    ax.legend(fontsize=9)

    lines = [
        f'VORHERSAGE',
        f'  Ort   : {pred_geo["city"]}, {pred_geo["country"]}',
        f'  Koord : {pred_lat:.1f}°N  {pred_lon:.1f}°E',
        f'  Konfidenz: {100*top_probs[0]:.1f}%',
    ]
    if true_geo:
        lines += [
            f'WAHRHEIT',
            f'  Ort   : {true_geo["city"]}, {true_geo["country"]}',
            f'  Koord : {true_lat:.1f}°N  {true_lon:.1f}°E',
            f'  Abstand: {dist_km:,.0f} km',
        ]
    ax.set_title('\n'.join(lines), fontsize=9, loc='left', family='monospace')

    plt.tight_layout(); plt.show()

    print('\nTop-K Vorhersagen:')
    for rank, (idx, prob) in enumerate(zip(top_idxs, top_probs), 1):
        plat, plon = cell_center(idx_to_cell[idx])
        geo = reverse_geocode(plat, plon)
        print(f'  {rank}. {geo["city"]}, {geo["country"]}  '
              f'({plat:.1f}°N {plon:.1f}°E)  {100*prob:.1f}%')

    return {'pred_lat': pred_lat, 'pred_lon': pred_lon,
            'pred_country': pred_geo['country'], 'pred_city': pred_geo['city'],
            'confidence': float(top_probs[0]), 'distance_km': dist_km}


# Demo mit einem zufälligen Test-Bild
sample = df_test.sample(1, random_state=7).iloc[0]
result = predict_image(sample['img_path'], sample['lat'], sample['lon'])

## 13. Massen-Test: 16 Bilder auf einmal

In [ ]:
N_SHOW   = 16
samples  = df_test.sample(N_SHOW, random_state=SEED)
results  = []

print('Vorhersagen berechnen...')
model.eval()
for _, row in tqdm(samples.iterrows(), total=N_SHOW):
    img = Image.open(row['img_path']).convert('RGB')
    t   = val_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_idx = model(t).argmax(dim=1).item()
    p_lat, p_lon = cell_center(idx_to_cell[pred_idx])
    results.append({
        'img_path'    : row['img_path'],
        'true_lat'    : row['lat'],   'true_lon'    : row['lon'],
        'pred_lat'    : p_lat,        'pred_lon'    : p_lon,
        'distance_km' : haversine_km(p_lat, p_lon, row['lat'], row['lon']),
    })

print('Reverse Geocoding...')
for r in tqdm(results):
    r['pred_geo'] = reverse_geocode(r['pred_lat'], r['pred_lon'])
    r['true_geo'] = reverse_geocode(r['true_lat'], r['true_lon'])

df_results = pd.DataFrame(results)
df_results['pred_country'] = df_results['pred_geo'].apply(lambda g: g['country'])
df_results['true_country'] = df_results['true_geo'].apply(lambda g: g['country'])
df_results['pred_city']    = df_results['pred_geo'].apply(lambda g: g['city'])
df_results['true_city']    = df_results['true_geo'].apply(lambda g: g['city'])

country_acc = (df_results['true_country'] == df_results['pred_country']).mean()
print(f'\nLänder-Genauigkeit  : {100*country_acc:.0f}%')
print(f'Median-Distanz      : {df_results["distance_km"].median():,.0f} km')
df_results[['true_country','pred_country','true_city','pred_city','distance_km']]

In [ ]:
# Bildgitter
cols   = 4
rows_n = math.ceil(N_SHOW / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(cols*4, rows_n*3.5))

for ax, (_, r) in zip(axes.flat, df_results.iterrows()):
    try:
        ax.imshow(Image.open(r['img_path']).convert('RGB'))
    except Exception:
        ax.text(0.5, 0.5, 'Fehler', ha='center')
    km    = r['distance_km']
    color = 'green' if km < 750 else ('orange' if km < 2500 else 'red')
    ax.set_title(
        f'Wahr: {r["true_city"][:16]}, {r["true_country"][:12]}\n'
        f'Pred: {r["pred_city"][:16]}, {r["pred_country"][:12]}\n'
        f'{km:,.0f} km',
        fontsize=6.5, color=color
    )
    ax.axis('off')

for ax in axes.flat[N_SHOW:]:
    ax.axis('off')

plt.suptitle('Ergebnisse: grün <750km  orange <2500km  rot >2500km', fontsize=11)
plt.tight_layout(); plt.show()

## 14. Interaktive Weltkarte (Folium)

In [ ]:
import folium

m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB positron')

for _, r in df_results.iterrows():
    km    = r['distance_km']
    color = 'green' if km < 750 else ('orange' if km < 2500 else 'red')

    folium.CircleMarker(
        [r['true_lat'], r['true_lon']], radius=7,
        color='blue', fill=True, fill_opacity=0.85,
        popup=f"Wahrheit: {r['true_city']}, {r['true_country']}",
    ).add_to(m)

    folium.CircleMarker(
        [r['pred_lat'], r['pred_lon']], radius=7,
        color=color, fill=True, fill_opacity=0.85,
        popup=f"Vorhersage: {r['pred_city']}, {r['pred_country']} | {km:,.0f} km",
    ).add_to(m)

    folium.PolyLine(
        [[r['true_lat'], r['true_lon']], [r['pred_lat'], r['pred_lon']]],
        color='gray', weight=1.5, opacity=0.5,
    ).add_to(m)

legend = '''
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
            padding:10px;border-radius:6px;font-size:12px;border:1px solid #ccc">
  <b>Legende</b><br>
  🔵 Wahre Position<br>
  🟢/🟠/🔴 Vorhersage (nach Distanz)<br>
  ── Abweichung
</div>
'''
m.get_root().html.add_child(folium.Element(legend))
m.save('geo_predictions_map.html')
print('Karte gespeichert: geo_predictions_map.html')
m

## 15. Eigenes Bild testen (Standalone-Inference)

Nur diesen Block ausführen – kein Retraining nötig.

In [ ]:
# ─────────────── HIER ANPASSEN ───────────────────────────────────
MY_IMG  = r'C:\Pfad\zu\deinem\bild.jpg'   # Pfad zum eigenen Bild
MY_LAT  = None   # optional: echte Koordinaten (z.B. 47.37 für Zürich)
MY_LON  = None   # optional: (z.B. 8.54)
# ─────────────────────────────────────────────────────────────────

if not Path(CFG['best_model']).exists():
    print('Kein Modell gefunden – bitte zuerst Training (Abschnitt 9) ausführen.')
elif not Path(MY_IMG).exists():
    print(f'Bilddatei nicht gefunden: {MY_IMG}')
    print('Demo mit zufälligem Test-Bild:')
    s = df_test.sample(1, random_state=99).iloc[0]
    predict_image(s['img_path'], s['lat'], s['lon'])
else:
    predict_image(MY_IMG, MY_LAT, MY_LON)

---
## Zusammenfassung

| Schritt | Was passiert |
|---------|-------------|
| **Download** | 10.000 Street-View-Bilder + GPS-Koordinaten |
| **Gitter** | Welt in `grid_deg`×`grid_deg` Zellen = Klassen |
| **Modell** | EfficientNet-B0 pretrained + eigener Kopf |
| **Training** | CrossEntropy + AdamW + Cosine LR Scheduler |
| **Metrik** | Haversine-Distanz km, GCD-Kurve |
| **Geocoding** | Zellmittelpunkt → Ortsname via Nominatim |
| **Ausgabe** | Land, Stadt, Karte, km-Abweichung |

**Erwartbare Ergebnisse (10k Bilder, 10° Grid, CPU):**
- Kontinent-Genauigkeit: ~40–60%
- Land-Genauigkeit: ~15–30%
- Median-Distanz: ~3.000–6.000 km

**Für bessere Ergebnisse:**
- Kleineres Grid (5°) + mehr Daten
- Größerer Backbone (b4, b7)
- GPU + mehr Epochen